# Two-body low-energy scattering
**Pedro Henrique Gesualdo Modesto** — IFSC/USP · advisor Lucas Madeira

At low energy only two numbers survive, the scattering length $a$ and the
effective range $r_0$:  $k\cot\delta_0 = -1/a + \tfrac12 r_0k^2 + O(k^4)$.

**Question: tuned to the same $(a,r_0)$, do unrelated potentials predict the
same binding energy?**

Two files only: this notebook and `lab.py` beside it. Units there are
$\hbar=\mu_{red}=1$ and lengths in fm, so $u''=2(V-E)u$.

Run All, about 60 s.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lab

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})

# One colour per potential, reused in every figure.
COLOR = {"well": "#1f77b4", "mpt": "#d62728",
         "gauss": "#2ca02c", "lj": "#9467bd"}

for tag in lab.SOURCES:
    print(tag, lab.SOURCES[tag])

# hbar^2/2mu is not tabulated anywhere; lab.py gets it by inverting the
# published zero-range energy. Recovering the known constants checks the table.
print()
for name in lab.SYSTEMS:
    print("hbar^2/2mu for", name, "=", round(lab.SYSTEMS[name]["h2_2mu"], 3))

## 1. Four potentials at unitarity

Square well (discontinuous, Eq. 70), Pöschl-Teller (smooth and solvable,
Eq. 116), Gaussian (tail dies faster than exponentially, Eq. 120),
Lennard-Jones (hard core plus van der Waals tail, Eq. 121).

At unitarity $|a|\to\infty$, so $a$ is not comparable between implementations —
$1/a=0$ is. The real test is $r_0$, which must come out 1 fm.

> **A factor of 2 in Eq. (121).** The Lennard-Jones is written in the article as
> $V = (\hbar^2/m_r)(C_{12}/r^{12} - C_6/r^6)$, with no $\tfrac12$. Taken
> literally it does not reproduce Table 4 of the same paper. For the deuteron
> row, whose published parameters are $C_{12}=0.90485319$ and $C_6=6.81472$ and
> which Table 4 reports as $a = 5.4$ fm:
>
> | | $a$ (fm) | $r_0$ (fm) |
> |---|---|---|
> | Table 4 | 5.4 | 1.70 |
> | with $\tfrac12$ | 5.405 | 1.699 |
> | without | **1.435** | 1.412 |
>
> Eqs. (70) and (120) carry the same $\hbar^2/m_r$ prefactor and do reproduce
> Table 3 as printed, so the mismatch is specific to Eq. (121). The likely
> reading is $2m_r$ in its denominator. `lab.py` uses the $\tfrac12$.

In [ ]:
# Build the four potentials with the parameters published for unitarity.
pots = {}
for name in lab.POTENTIALS:
    p = lab.PUBLISHED[("unitarity", name)]
    pots[name] = lab.POTENTIALS[name](p["p1"], p["p2"])

fig, ax = plt.subplots(1, 2, figsize=(10, 3), layout="tight")

# Left: the three smooth potentials share a linear scale.
r = np.linspace(0.001, 4, 500)
for name in ["well", "mpt", "gauss"]:
    ax[0].plot(r, pots[name].V(r), color=COLOR[name], lw=1.5, label=pots[name].name)
ax[0].set(xlabel="r (fm)", ylabel="V (fm$^{-2}$)", title="Fig. 1a - smooth")
ax[0].legend(fontsize=7)

# Right: the Lennard-Jones core reaches 1e10, so it needs log axes.
r = np.logspace(-2, 0.5, 500)
ax[1].plot(r, pots["lj"].V(r), color=COLOR["lj"], lw=1.5)
ax[1].set(xscale="log", yscale="symlog", xlabel="r (fm)",
          title="Fig. 1b - Lennard-Jones")
plt.show()

# Measure all four.
rows = []
for name in pots:
    a, r0, nodes = lab.scattering(pots[name])
    rows.append({"potential": pots[name].name, "R": pots[name].R,
                 "a": a, "r0": r0, "nodes": nodes})
display(pd.DataFrame(rows))

## 2. The inverse problem: automatic tuning

Given a target $(a, r_0)$, find the parameters. Two nested loops, each solving
one equation in one variable: the inner one moves the strength until $1/a$
matches, the outer one moves the scale until $r_0$ matches.

**We chase $1/a$, never $a$.** $a$ has poles — one every time a new bound
state appears — and bisection dies on a pole but works fine on a zero.
Unitarity is then just $1/a = 0$, with no special case.

Counting nodes is not optional: two solutions can share $(a, r_0)$ and differ
in node count, and then they are not the same physical state.

Below: all of Tables 3 and 4 of the article, 3 targets times 4 potentials.
Each run starts from the published parameters and has to return to them.

In [ ]:
rows = []
for case in lab.TARGETS:
    target = lab.TARGETS[case]
    for name in lab.POTENTIALS:
        pub = lab.PUBLISHED[(case, name)]
        strength, scale, a, r0, nodes, ok = lab.tune(
            name, target["a"], target["r0"], pub["p1"], pub["p2"],
            nodes_target=target["nodes"])

        # How far the two parameters moved from the published values.
        off_1 = abs(strength / pub["p1"] - 1)
        off_2 = abs(scale / pub["p2"] - 1)

        rows.append({"case": case, "potential": lab.POTENTIALS[name].name,
                     "p1": strength, "p1_pub": pub["p1"],
                     "p2": scale, "p2_pub": pub["p2"],
                     "r0": r0, "r0_pub": pub["r0_pub"],
                     "nodes": nodes, "ok": ok,
                     "dev_pct": 100 * max(off_1, off_2)})

tuning = pd.DataFrame(rows)
display(tuning)

Ten of the twelve land within 0.2%. Two do not: Lennard-Jones/nn and
Pöschl-Teller/deuteron, both near 2.7%. Refining the tolerance does not move
them, and **a discrepancy that survives refinement is never numerical.** The
`r0_pub` column is the reason, and it is `lab.SOURCES["D1"]`: the article's own
parameters give $r_0 = 2.71$ and $1.73$ where its Table 2 lists the targets as
$2.70$ and $1.70$. We tune to the target, so we land elsewhere — correctly.

### What universality looks like

The three potentials above have nothing in common inside their range — one is
discontinuous, one is smooth, one is a Gaussian. Tuned to the same
$(a, r_0)$, their zero-energy wavefunctions **coincide outside the range and
differ completely inside it**. Everything a low-energy experiment can see
lives outside. That is the whole claim of this notebook, in one picture.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4), layout="tight")

for name in ["well", "mpt", "gauss"]:
    row = tuning[(tuning["case"] == "deuteron") &
                 (tuning["potential"] == lab.POTENTIALS[name].name)].iloc[0]
    pot = lab.POTENTIALS[name](row["p1"], row["p2"])
    r, u, x, h, slope, r_start = lab.numerov(pot, 0.0)
    a = pot.R - u[-1] / slope
    u = u * (1.0 - pot.R / a) / u[-1]         # same normalisation as scattering()
    ax.plot(r, u, color=COLOR[name], lw=1.5, label=pot.name)
    ax.axvline(pot.R, color=COLOR[name], ls=":", lw=0.8)   # where each one ends

# The asymptote they all share: the straight line 1 - r/a.
r_line = np.linspace(0, 9, 50)
ax.plot(r_line, 1 - r_line / a, "k--", lw=1.3, label="asymptote $1 - r/a$")
ax.plot([a], [0], "ko", ms=6)
ax.annotate("a = %.2f fm" % a, (a, 0), (a - 2.6, 0.35), fontsize=8,
            arrowprops={"arrowstyle": "->", "lw": 0.9})
ax.axhline(0, color="k", lw=0.6)
ax.set(xlim=(0, 9), xlabel="r (fm)", ylabel="u(r) at E = 0",
       title="Fig. 3 - tuned to the deuteron: identical outside, different inside\n"
             "(dotted lines mark where each potential ends)")
ax.legend(fontsize=7)
plt.show()

## 3. Bound states

Two real systems nine orders of magnitude apart in energy, every potential on
the same $(a, r_0)$, compared against two formulas:

- **zero range**, $E = -(\hbar^2/2\mu)/a^2$, which throws away the size of
  the potential entirely;
- **finite range**, the root of $\kappa = 1/a + r_0\kappa^2/2$, which keeps
  $r_0$ and drops only $O(k^4)$.

Lennard-Jones is skipped for helium: with $a = 90$ Å its tail only dies after
thousands of Å.

In [ ]:
def run(system_name, names):
    """Tune every potential on this system, then solve for its bound state."""
    s = lab.SYSTEMS[system_name]
    h2 = s["h2_2mu"]
    scale_factor = s["r0"] / 1.7436          # lengths relative to the deuteron

    E_finite = lab.E_finite_range(s["a"], s["r0"], h2)
    rows = [{"method": "experiment", "E": s["E"]},
            {"method": "zero range", "E": lab.E_zero_range(s["a"], h2)},
            {"method": "finite range", "E": E_finite}]

    for name in names:
        strength, scale = lab.GUESS[name]
        # Rescale the starting guess: C12 carries L^6, every other scale 1/L.
        if name == "lj":
            scale = scale * scale_factor ** 6
        else:
            scale = scale / scale_factor

        strength, scale, a, r0, nodes, ok = lab.tune(
            name, s["a"], s["r0"], strength, scale, nodes_target=1)

        pot = lab.POTENTIALS[name](strength, scale)
        E_code = lab.bound_energy(pot, E_finite / (2 * h2))
        rows.append({"method": pot.name, "E": 2 * h2 * E_code})

    out = pd.DataFrame(rows)
    out["dev_pct"] = 100 * (out["E"] / s["E"] - 1)
    return out


results = {"deuteron": run("deuteron", ["well", "mpt", "gauss", "lj"]),
           "he4_dimer": run("he4_dimer", ["well", "mpt", "gauss"])}

fig, ax = plt.subplots(1, 2, figsize=(10, 3), layout="tight")
for k, system_name in enumerate(results):
    s = lab.SYSTEMS[system_name]
    table = results[system_name]
    print()
    print(system_name, " |a|/r0 =", round(abs(s["a"]) / s["r0"], 1),
          " unit:", s["unit"])
    display(table)

    # Bar chart of the deviation, dropping the experimental row (it is the zero).
    bars = table[table["method"] != "experiment"]
    colors = ["#888888", "#333333"]
    for name in ["well", "mpt", "gauss", "lj"]:
        if len(colors) < len(bars):
            colors.append(COLOR[name])
    ax[k].bar(bars["method"], bars["dev_pct"], color=colors)
    ax[k].axhspan(-1, 1, color="green", alpha=0.12)      # the +/- 1% band
    ax[k].axhline(0, color="k", lw=1)
    ax[k].tick_params(axis="x", rotation=35, labelsize=7)
    ax[k].set(ylabel="deviation (%)", title="Fig. 2%s - %s" % ("ab"[k], system_name))
plt.show()

Zero range misses by **36% for the deuteron** and 8.6% for helium. The
difference is $|a|/r_0$: 3.1 against 11.3. **The deuteron — the textbook
shallow bound state — fails the usual criterion $|a|/r_0 > 10$.** Adding
$r_0$ brings both under 1%. The leftover spread of about 1% between
potentials *is* the shape dependence, that is, the $O(k^4)$ terms.

## 4. Reproducing Figs. 5 and 6 of the article

The article uses dimensionless axes, $a/R$ and $r_0/R$ against
$\sqrt{2v_0} = k_0R$, and that is what puts the poles exactly at
$\pi/2 + n\pi$ with nothing fitted.

The sharp test is Fig. 6: at every pole of $a$ the article predicts
$r_0/R = 1$ **exactly** — the effective range equals the range of the
potential.

In [ ]:
# x = sqrt(2 v0) is the article's axis, so v0 = x^2/2.
x = np.linspace(0.02, 11, 4000)
a_exact = []
r0_exact = []
for value in x:
    v = value ** 2 / 2
    a_exact.append(lab.a_well(v))
    r0_exact.append(lab.r0_well(v))
a_exact = np.array(a_exact)
r0_exact = np.array(r0_exact)

# Two different families of singularities, and they do not coincide:
poles = np.pi / 2 + np.arange(4) * np.pi      # a diverges: a new bound state
zeros = [4.4934, 7.7253, 10.9041]             # tan x = x, so a = 0 and r0 diverges

# Check points for the solver, chosen to stay clear of both families.
x_check = [0.5, 1, 2, 2.5, 3, 3.5, 4, 5, 5.5, 6, 6.5, 7, 8.5, 9, 9.5, 10.5]
a_num = []
r0_num = []
for value in x_check:
    a, r0, nodes = lab.scattering(lab.Well(value ** 2 / 2, 1.0))
    a_num.append(a)
    r0_num.append(r0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), layout="tight")

# Blank out the huge values so matplotlib does not draw a line across each pole.
for k, curve in enumerate([a_exact, r0_exact]):
    drawable = np.where(abs(curve) > 8, np.nan, curve)
    ax[k].plot(x, drawable, "k", lw=1.3, label="closed form")
    for p in poles:
        ax[k].axvline(p, color="gray", ls="--", lw=0.8)
    ax[k].axhline(0, color="k", lw=0.5)
    ax[k].legend(fontsize=7, loc="lower right")

ax[0].plot(x_check, a_num, "o", ms=4, color=COLOR["well"], label="this solver")
ax[1].plot(x_check, r0_num, "o", ms=4, color=COLOR["well"], label="this solver")
ax[1].plot(poles, [1, 1, 1, 1], "r*", ms=11, zorder=5)   # the exact prediction
for q in zeros:
    ax[1].axvline(q, color="orange", ls="-.", lw=0.8)

ax[0].set(xlim=(0, 11), ylim=(-4, 4), xlabel=r"$\sqrt{2v_0}$", ylabel="$a/R$",
          title=r"Fig. 3a - article Fig. 5: poles at $\pi/2+n\pi$")
ax[1].set(xlim=(0, 11), ylim=(-3, 3), xlabel=r"$\sqrt{2v_0}$", ylabel="$r_0/R$",
          title=r"Fig. 3b - article Fig. 6: red stars are $r_0/R=1$")
plt.show()

# The number behind the red stars.
print("r0/R at the poles of a (the article says exactly 1):")
for p in poles:
    print("   sqrt(2v0) =", round(p, 6), " ->  r0/R =", round(lab.r0_well(p ** 2 / 2), 10))

# And the solver against the closed forms, away from the poles.
worst_a = 0.0
worst_r0 = 0.0
for i in range(len(x_check)):
    v = x_check[i] ** 2 / 2
    worst_a = max(worst_a, abs(a_num[i] - lab.a_well(v)))
    worst_r0 = max(worst_r0, abs(r0_num[i] - lab.r0_well(v)))
print("worst error over %d points:  a/R %.1e   r0/R %.1e" % (len(x_check), worst_a, worst_r0))

## 5. Is the grid fine enough?

A calculation that reproduces an article but never checks its own grid is
saying "it ran", not "it is right". Halving the points must change nothing.

This is not a formality: it is what caught the one real bug in this code. On
a uniform grid the Lennard-Jones $r_0$ went 1.74257 → 1.74215 → 1.74165 as
the points doubled — drifting, not converging. Its hard core needs a step
0.002 fm wide while its $1/r^6$ tail needs a grid 125 fm long, and no single
uniform step serves both. The logarithmic grid does.

In [ ]:
rows = []
for name in lab.POTENTIALS:
    pub = lab.PUBLISHED[("deuteron", name)]
    pot = lab.POTENTIALS[name](pub["p1"], pub["p2"])
    a_fine, r0_fine, nodes = lab.scattering(pot)

    lab.POINTS = (lab.POINTS - 1) // 2 + 1          # half the points
    a_half, r0_half, nodes_half = lab.scattering(lab.POTENTIALS[name](pub["p1"], pub["p2"]))
    lab.POINTS = 2 * (lab.POINTS - 1) + 1           # put it back

    rows.append({"potential": pot.name, "points": lab.POINTS,
                 "a": a_fine, "a_shift": abs(a_fine / a_half - 1),
                 "r0": r0_fine, "r0_shift": abs(r0_fine / r0_half - 1)})

display(pd.DataFrame(rows))
print("A shift of 1e-11 is machine precision. The Lennard-Jones sits at 1e-7")
print("because its core and its tail differ by five orders of magnitude in scale.")
print()
print("Run test_lab.py for the full set of checks against closed forms.")

## Summary

1. **The tuning works for any potential** — 12 fits, 10 within 0.2%, and the
   other two explained by the reference itself.
2. **Two numbers describe low-energy physics** — unrelated potentials on the
   same $(a, r_0)$ agree to about 1% across nine orders of magnitude.
3. **The effective range is not a detail** — zero range misses the deuteron
   by 36%, and $|a|/r_0 = 3.1$ says why.
4. **The square well comes out exact** — $r_0/R = 1$ at every pole to ten
   digits, and the solver tracks the closed forms to $10^{-9}$.

**Next:** three bodies, which need exactly this — a two-body potential
anchored on known $(a, r_0)$. Quantum Monte Carlo will take these as input.

**Out of scope on purpose:** no error bars, no Lennard-Jones for helium,
s-wave only.